In [ ]:
import pandas as pd
import rdkit.Chem as Chem

from config import SplitType, TrainConfig
from models import chemprop_modded_ref as cpm_ref
from models import chemprop_ref as cp_ref
from preprocessing import get_butina_clusters, mol_to_inchi, standardize
from train import train_and_evaluate

In [ ]:
df = pd.read_csv("./datasets/ADME_public_set_3521.csv")
df = df.loc[:, ["SMILES", "LOG HLM_CLint (mL/min/kg)"]]
df.columns = ["smiles", "target"]
df = df.dropna(subset="target").reset_index(drop=True)

df["mol"] = df["smiles"].map(standardize)
df["inchi"] = df["mol"].map(mol_to_inchi)
df["mol"] = df["inchi"].map(Chem.MolFromInchi)
df["butina_cluster"] = get_butina_clusters(df["mol"])

In [ ]:
res_cp = train_and_evaluate(
    df, SplitType.BUTINA, cp_ref.train_and_evaluate_on_split, TrainConfig(max_epochs=40)
)

In [ ]:
res_cpm = train_and_evaluate(
    df,
    SplitType.BUTINA,
    cpm_ref.train_and_evaluate_on_split,
    TrainConfig(max_epochs=40),
)